- ## What is a Transformer block?
- A Transformer block is a collection of neural network components that work together to process token representations.
#
- #### Your ModernGPT uses these main components:
1. RMSNorm
2. Multi-Head Self-Attention
3. Residual connections
4. RMSNorm
5. SwiGLU Feed-Forward Network
63. Residual connections

- One block processes the input and produces an output with the same shape.

- Understand the complete flow
                Input x
                   |
             +-----+-------------------+
             |                         |
             v                         |
          RMSNorm                      |
             |                         |
             v                         |
      Multi-Head Attention             |
             |                         |
             +------------+------------+
                          |
                    Residual Add
                          |
                          v
                         x1
                          |
             +------------+------------+
             |                         |
             v                         |
          RMSNorm                      |
             |                         |
             v                         |
           SwiGLU                      |
             |                         |
             +------------+------------+
                          |
                    Residual Add
                          |
                          v
                       Output

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# -------------------------
# 1. RMSNorm
# -------------------------

class RMSNorm(nn.Module):

    def __init__(self, embed_dim, eps=1e-8):
        super().__init__()

        self.eps = eps # eps=1e-8: വളരെ ചെറിയ സംഖ്യ. Zero division അല്ലെങ്കിൽ വളരെ ചെറിയ denominator മൂലമുള്ള numerical instability കുറയ്ക്കാൻ ഉപയോഗിക്കുന്നു.
        self.gamma = nn.Parameter(torch.ones(embed_dim))

    def forward(self, x):

        mean_squared = x.pow(2).mean(
            dim=-1, # dim=-1 എന്നത് E-യെ സൂചിപ്പിക്കുന്നു. അതിനാൽ ഓരോ token-ന്റെയും embedding values-ലാണ് RMS കണക്കാക്കുന്നത്; വ്യത്യസ്ത tokens-നെ ഒരുമിച്ച് average ചെയ്യുന്നില്ല.
            keepdim=True # average കണക്കാക്കുമ്പോൾ ആ dimension-ന്റെ വലുപ്പം 1 ആയി നിലനിർത്തുന്നു.
        )

        rms = torch.sqrt(mean_squared + self.eps)

        normalized_x = x / rms

        return normalized_x * self.gamma


# -------------------------
# 2. Rotary Positional Embedding
# -------------------------

def apply_rope(x):

    # x shape: [B, H, T, D]
    batch_size, num_heads, seq_len, head_dim = x.shape

    # RoPE requires an even head dimension
    assert head_dim % 2 == 0 # check head dim is even

    device = x.device # Tensor CPU-യിലാണോ GPU-യിലാണോ എന്ന് കണ്ടെത്തുന്നു.
    dtype = x.dtype # Tensor-ന്റെ datatype കണ്ടെത്തുന്നു; ഉദാഹരണത്തിന് torch.float32.

    positions = torch.arange(
        seq_len,
        device=device,
        dtype=torch.float32
    )

    frequencies = 1.0 / (10000 ** (torch.arange(0,head_dim,2,device=device,dtype=torch.float32) / head_dim)) # which is also = 1/10000**(0,2,4,...62)/64
    angles = positions[:, None] * frequencies[None, :]

    cos = angles.cos()[None, None, :, :].to(dtype)
    sin = angles.sin()[None, None, :, :].to(dtype)

    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    rotated_even = x_even * cos - x_odd * sin
    rotated_odd = x_even * sin + x_odd * cos

    rotated = torch.stack(
        [rotated_even, rotated_odd],
        dim=-1
    )

    return rotated.flatten(-2)


# -------------------------
# 3. Multi-Head Self-Attention
# -------------------------

class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.query_layer = nn.Linear(
            embed_dim, embed_dim, bias=False
        )

        self.key_layer = nn.Linear(
            embed_dim, embed_dim, bias=False
        )

        self.value_layer = nn.Linear(
            embed_dim, embed_dim, bias=False
        )

        self.output_layer = nn.Linear(
            embed_dim, embed_dim, bias=False
        )

    def forward(self, x):

        batch_size, seq_len, embed_dim = x.shape

        Q = self.query_layer(x)
        K = self.key_layer(x)
        V = self.value_layer(x)

        Q = Q.reshape(
            batch_size, seq_len,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        K = K.reshape(
            batch_size, seq_len,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        V = V.reshape(
            batch_size, seq_len,
            self.num_heads, self.head_dim
        ).transpose(1, 2)

        # Apply RoPE to queries and keys
        Q = apply_rope(Q)
        K = apply_rope(K)

        # Causal self-attention
        output = F.scaled_dot_product_attention(
            Q,
            K,
            V,
            is_causal=True
        )

        output = output.transpose(1, 2).contiguous()

        output = output.reshape(
            batch_size, seq_len, self.embed_dim
        )

        return self.output_layer(output)


# -------------------------
# 4. SwiGLU
# -------------------------

class SwiGLU(nn.Module):

    def __init__(self, embed_dim, hidden_dim):
        super().__init__()

        self.gate_proj = nn.Linear(
            embed_dim, hidden_dim, bias=False
        )

        self.up_proj = nn.Linear(
            embed_dim, hidden_dim, bias=False
        )

        self.down_proj = nn.Linear(
            hidden_dim, embed_dim, bias=False
        )

    def forward(self, x):

        gate = F.silu(self.gate_proj(x))

        up = self.up_proj(x)

        combined = gate * up

        return self.down_proj(combined)


# -------------------------
# 5. Transformer Block
# -------------------------

class TransformerBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        hidden_dim
    ):
        super().__init__()

        self.norm1 = RMSNorm(embed_dim)

        self.attention = MultiHeadAttention(
            embed_dim,
            num_heads
        )

        self.norm2 = RMSNorm(embed_dim)

        self.feed_forward = SwiGLU(
            embed_dim,
            hidden_dim
        )

    def forward(self, x):

        # Attention + residual connection
        x = x + self.attention(self.norm1(x))

        # SwiGLU + residual connection
        x = x + self.feed_forward(self.norm2(x))

        return x


# -------------------------
# 6. Test the Transformer Block
# -------------------------

if __name__ == "__main__":

    batch_size = 16
    seq_len = 128
    embed_dim = 256
    num_heads = 4
    hidden_dim = 1024

    x = torch.randn(
        batch_size,
        seq_len,
        embed_dim
    )

    model = TransformerBlock(
        embed_dim=embed_dim,
        num_heads=num_heads,
        hidden_dim=hidden_dim
    )

    output = model(x)

    print("Input shape:", x.shape)
    print("Output shape:", output.shape)

Input shape: torch.Size([16, 128, 256])
Output shape: torch.Size([16, 128, 256])
